# directory-pipeline — Chandra OCR test

Runs Chandra OCR (`datalab-to/chandra-ocr-2`, 5B params) on a selected volume
and optionally aligns the output with Surya bounding boxes.

**Before you start:**
- Enable a GPU runtime: Runtime → Change runtime type → **T4 GPU**  
  (the 5B model needs ~10 GB VRAM; T4 has 16 GB)
- Run the **Setup** cells first, then jump to whichever section you need.

**Typical workflow for a first test:**
1. Run all Setup cells
2. Pick a small sub-volume in the *Configure* cell (~30–50 pages is a good first test)
3. Run *Install Chandra* then *Run Chandra OCR*
4. Inspect a few `.txt` output files to check quality
5. Optionally run *Align OCR* and compare with an existing Gemini alignment

---
## Setup

Run these cells once per session.

In [ ]:
# --- Static config — edit once ---
PIPELINE_DIR = "/content/drive/Othercomputers/My_Mac/directory-pipeline"

# Volume to test — set a sub-volume (UUID dir) for large collections.
# Examples:
#   "output/green_books_and_related/the_green_book_9ea5d5b0/fc1bcc10-1a76-0132-8925-58d385a7bbd0"
#   "output/green_books_and_related/travelguide_634f3af0/7f6fa2d0-bddf-0136-1cf5-0678fe8fb234"
VOLUME = "output/green_books_and_related/the_green_book_9ea5d5b0/fc1bcc10-1a76-0132-8925-58d385a7bbd0"

# Chandra batch size — 1 is safest on T4; increase to 2–4 if you have VRAM headroom
BATCH_SIZE = 1

# Gemini model slug already used on this volume (for alignment comparison)
GEMINI_MODEL = "gemini-2.0-flash"

In [ ]:
from google.colab import drive
import os

drive.mount("/content/drive")
os.chdir(PIPELINE_DIR)
print("Working directory:", os.getcwd())

In [ ]:
# Install base pipeline dependencies (skip if already installed)
%pip install -q google-genai pillow requests python-dotenv flask geopy pyngrok

In [ ]:
# Install Chandra OCR with HuggingFace backend (~2–3 min first time; pulls torch + transformers)
%pip install -q "chandra-ocr[hf]"

# Quick smoke-test: verify chandra can be imported
from chandra.model.schema import BatchInputItem
print("chandra-ocr imported OK")

In [ ]:
import torch
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    total_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"VRAM: {total_gb:.1f} GB")
    if total_gb < 10:
        print("WARNING: <10 GB VRAM — Chandra (5B BF16 ~10 GB) may OOM. Try batch-size 1.")
    else:
        print("OK — enough VRAM for Chandra BF16")

---
## Volume info

Check how many images are in the selected volume and how many have already been
processed by Chandra or Gemini.

In [ ]:
from pathlib import Path

vol = Path(PIPELINE_DIR) / VOLUME
assert vol.exists(), f"Volume not found: {vol}"

jpgs = sorted(p for p in vol.rglob("*.jpg") if not p.stem.endswith("_viz"))

chandra_done = len(list(vol.rglob("*_chandra-ocr-2.txt")))
gemini_done  = len(list(vol.rglob(f"*_{GEMINI_MODEL}.txt")))
aligned_gemini = len(list(vol.rglob(f"*_{GEMINI_MODEL}_aligned.json")))
aligned_chandra = len(list(vol.rglob("*_chandra-ocr-2_aligned.json")))

print(f"Volume:          {vol}")
print(f"Total images:    {len(jpgs)}")
print(f"Chandra .txt:    {chandra_done} / {len(jpgs)}")
print(f"Gemini .txt:     {gemini_done} / {len(jpgs)}")
print(f"Chandra aligned: {aligned_chandra} / {len(jpgs)}")
print(f"Gemini aligned:  {aligned_gemini} / {len(jpgs)}")

if chandra_done == len(jpgs):
    print("\n✓ All images already processed by Chandra — skip to Inspect or Align.")
else:
    remaining = len(jpgs) - chandra_done
    # Very rough estimate: ~20s/image on T4 at batch_size=1
    est_min = remaining * 20 / 60
    print(f"\nEstimated time: ~{est_min:.0f} min for {remaining} remaining images at ~20s/image (T4, batch=1)")

---
## Run Chandra OCR

Loads the 5B model once (~30 s) then processes images sequentially.
Output: `{stem}_chandra-ocr-2.txt` alongside each image.
Already-processed images are skipped, so this cell is safe to re-run.

In [ ]:
!time python pipeline/run_chandra_ocr.py {VOLUME} --batch-size {BATCH_SIZE}

---
## Inspect output

Compare Chandra and Gemini plain-text output side-by-side for a few pages.
This is the most important check before running alignment.

In [ ]:
from pathlib import Path
from IPython.display import display, HTML

vol = Path(PIPELINE_DIR) / VOLUME

# Find pages that have both Chandra and Gemini output
chandra_txts = sorted(vol.rglob("*_chandra-ocr-2.txt"))
sample = chandra_txts[:3]  # first 3; change slice to inspect different pages

if not sample:
    print("No Chandra output yet — run the OCR cell first.")
else:
    for ct in sample:
        gt = ct.parent / ct.name.replace("_chandra-ocr-2.txt", f"_{GEMINI_MODEL}.txt")
        chandra_text = ct.read_text(encoding="utf-8") if ct.exists() else "(not found)"
        gemini_text  = gt.read_text(encoding="utf-8") if gt.exists() else "(not found)"

        def _esc(s):
            return s.replace("&","&amp;").replace("<","&lt;").replace(">","&gt;")

        img_stem = ct.name.replace("_chandra-ocr-2.txt", "")
        display(HTML(f"""
        <h3 style="margin-top:24px">{img_stem}</h3>
        <table style="width:100%;border-collapse:collapse;font-family:monospace;font-size:12px">
          <tr>
            <th style="width:50%;background:#e8f4e8;padding:6px">Chandra</th>
            <th style="width:50%;background:#e8f0f8;padding:6px">Gemini ({GEMINI_MODEL})</th>
          </tr>
          <tr>
            <td style="padding:8px;vertical-align:top;border:1px solid #ccc;white-space:pre-wrap">{_esc(chandra_text[:3000])}</td>
            <td style="padding:8px;vertical-align:top;border:1px solid #ccc;white-space:pre-wrap">{_esc(gemini_text[:3000])}</td>
          </tr>
        </table>
        """))

In [ ]:
# Line count comparison — alignment works best when Chandra line count is
# close to Surya's line count.  Chandra >> Surya usually means extra blank
# lines or markdown structure; Chandra << Surya means merged lines.
from pathlib import Path

vol = Path(PIPELINE_DIR) / VOLUME
import json

rows = []
for ct in sorted(vol.rglob("*_chandra-ocr-2.txt"))[:20]:  # first 20 pages
    stem = ct.name.replace("_chandra-ocr-2.txt", "")
    chandra_lines = [l for l in ct.read_text(encoding="utf-8").splitlines() if l.strip()]

    gt = ct.parent / f"{stem}_{GEMINI_MODEL}.txt"
    gemini_lines = [l for l in gt.read_text(encoding="utf-8").splitlines() if l.strip()] if gt.exists() else []

    sj = ct.parent / f"{stem}_surya.json"
    surya_lines = len(json.loads(sj.read_text()).get("lines", [])) if sj.exists() else None

    rows.append((stem, len(chandra_lines), len(gemini_lines), surya_lines))

print(f"{'Page':<40} {'Chandra':>8} {'Gemini':>8} {'Surya':>8}")
print("-" * 68)
for stem, nc, ng, ns in rows:
    surya_str = str(ns) if ns is not None else "—"
    flag = "  ⚠ big diff" if ns and abs(nc - ns) > ns * 0.5 else ""
    print(f"{stem:<40} {nc:>8} {ng:>8} {surya_str:>8}{flag}")

---
## Align OCR

Aligns Chandra plain-text output with Surya bounding boxes using Needleman-Wunsch,
producing `{stem}_chandra-ocr-2_aligned.json` files.

Requires `*_surya.json` files — run Surya OCR first if they don't exist
(see `ocr-align-review.ipynb`).

> **Note**: `--model chandra-ocr-2` is required because `align_ocr.py` auto-detection
> only looks for `gemini-*` names; passing the model explicitly bypasses that.

In [ ]:
!time python pipeline/align_ocr.py {VOLUME} --model chandra-ocr-2

---
## Compare alignment quality

Check unmatched line counts for Chandra vs Gemini alignments.
Fewer unmatched lines = better alignment quality.

In [ ]:
import json
from pathlib import Path

vol = Path(PIPELINE_DIR) / VOLUME

def _alignment_stats(aligned_json: Path) -> dict:
    data = json.loads(aligned_json.read_text(encoding="utf-8"))
    lines = data.get("lines", [])
    total = len(lines)
    unmatched_g = len(data.get("unmatched_gemini", []))
    unmatched_s = sum(1 for l in lines if not l.get("gemini_text"))
    manual = sum(1 for l in lines if l.get("confidence") == "manual")
    return {"total": total, "unmatched_ocr": unmatched_g, "unmatched_surya": unmatched_s, "manual": manual}

chandra_jsons = sorted(vol.rglob("*_chandra-ocr-2_aligned.json"))
gemini_jsons  = sorted(vol.rglob(f"*_{GEMINI_MODEL}_aligned.json"))

if not chandra_jsons:
    print("No Chandra aligned JSON found — run the Align cell first.")
else:
    # Build lookup by image stem
    g_stats = {}
    for gj in gemini_jsons:
        stem = gj.name.replace(f"_{GEMINI_MODEL}_aligned.json", "")
        g_stats[stem] = _alignment_stats(gj)

    c_total_unmatched = 0
    g_total_unmatched = 0
    count = 0

    print(f"{'Page':<38} {'C unmatched':>12} {'G unmatched':>12} {'C lines':>8} {'G lines':>8}")
    print("-" * 80)
    for cj in chandra_jsons[:20]:  # first 20
        stem = cj.name.replace("_chandra-ocr-2_aligned.json", "")
        cs = _alignment_stats(cj)
        gs = g_stats.get(stem, {})
        g_un = gs.get("unmatched_ocr", "—")
        g_li = gs.get("total", "—")
        print(f"{stem:<38} {cs['unmatched_ocr']:>12} {str(g_un):>12} {cs['total']:>8} {str(g_li):>8}")
        c_total_unmatched += cs["unmatched_ocr"]
        if isinstance(g_un, int):
            g_total_unmatched += g_un
        count += 1

    print("-" * 80)
    print(f"{'TOTAL (shown pages)':<38} {c_total_unmatched:>12} {g_total_unmatched:>12}")
    if count:
        verdict = "Chandra ✓" if c_total_unmatched < g_total_unmatched else (
            "Gemini ✓" if g_total_unmatched < c_total_unmatched else "Tie")
        print(f"\nFewer unmatched lines: {verdict}")

---
## Review Chandra alignment

Start the interactive Flask review tool pointing at the Chandra-aligned files.
Same tool as the Gemini workflow — `--model chandra-ocr-2` tells it which
`*_aligned.json` files to load.

In [ ]:
import subprocess
from google.colab.output import eval_js

PORT = 5000
subprocess.run(["fuser", "-k", f"{PORT}/tcp"], capture_output=True)

subprocess.Popen(
    ["python", "-m", "pipeline.review_alignment",
     VOLUME,
     "--host", "0.0.0.0",
     "--port", str(PORT),
     "--model", "chandra-ocr-2"],
    cwd=PIPELINE_DIR,
    stdout=open("/tmp/review.log", "w"),
    stderr=subprocess.STDOUT,
    start_new_session=True,
)

import time; time.sleep(3)
print("Review URL:", eval_js(f"google.colab.kernel.proxyPort({PORT})"))

In [ ]:
# Watch server log — stop cell once you see "Models ready."
!tail -f /tmp/review.log

In [ ]:
# Optional: ngrok fallback if the proxy URL doesn't work
from pyngrok import ngrok
from google.colab import userdata
ngrok.set_auth_token(userdata.get("NGROK_TOKEN"))
for t in ngrok.get_tunnels():
    ngrok.disconnect(t.public_url)
print("Review URL (ngrok):", ngrok.connect(PORT))

---
## Debug: inspect raw Markdown before stripping

If alignment quality is poor, it can help to see what Chandra's raw Markdown
looked like before `_strip_markdown()` converted it to plain text.

In [ ]:
# Re-run Chandra on a single image and print the raw Markdown *before* stripping,
# then the stripped plain text, so you can see exactly what the aligner receives.
import sys, torch, time
from pathlib import Path
from PIL import Image
from transformers import AutoModelForImageTextToText, AutoProcessor
from chandra.model.hf import generate_hf
from chandra.model.schema import BatchInputItem
from chandra.output import parse_markdown
sys.path.insert(0, PIPELINE_DIR)
from pipeline.run_chandra_ocr import _strip_markdown

vol = Path(PIPELINE_DIR) / VOLUME
jpgs = sorted(p for p in vol.rglob("*.jpg") if not p.stem.endswith("_viz"))

# Pick an image to debug (change index as needed)
DEBUG_IMAGE = jpgs[0]
print("Image:", DEBUG_IMAGE.name)

# Load model (reuse if already in memory via _chandra_model global)
if "_chandra_model" not in dir():
    print("Loading model…")
    _chandra_model = AutoModelForImageTextToText.from_pretrained(
        "datalab-to/chandra-ocr-2", dtype=torch.bfloat16, device_map="auto"
    )
    _chandra_model.processor = AutoProcessor.from_pretrained("datalab-to/chandra-ocr-2")
    print("Model loaded.")

img = Image.open(DEBUG_IMAGE).convert("RGB")
result = generate_hf([BatchInputItem(image=img, prompt_type="ocr_layout")], _chandra_model)[0]
raw_md = parse_markdown(result.raw) if result.raw else (result.markdown or "")
plain  = _strip_markdown(raw_md)

print("\n" + "=" * 60)
print("RAW MARKDOWN (first 2000 chars):")
print("=" * 60)
print(raw_md[:2000])
print("\n" + "=" * 60)
print("STRIPPED PLAIN TEXT (first 2000 chars):")
print("=" * 60)
print(plain[:2000])